# COMS 6730 HW4: SFT TinyLlama for Disease NER (Nova OnDemand)

Fine-tune **TinyLlama 1.1B** (via Unsloth) to extract disease entities from sentences.

This is a **Nova / Open OnDemand** version of the Colab notebook.

## Launch Jupyter with a GPU

1. Go to [Nova OnDemand](https://nova-ondemand.its.iastate.edu)
2. **Interactive Apps → Jupyter**
3. Suggested form settings:
   - **Number of Tasks Per Node:** `4`–`6` (≤ 6 per GPU)
   - **GRES:** `gpu:1` (or `gpu:a100:1` / `gpu:v100:1` if needed)
   - **Memory:** `32G`–`64G`
   - **Time:** `2:00:00` or more
   - **Work Directory:** `/lustre/hdd/LAS/jernigan-lab/coms6730/HW4`
4. Connect → open this notebook → select your custom GPU kernel (created below)

Official docs: [JupyterLab on Nova](https://research.it.iastate.edu/jupyterlab) · [GPUs](https://research.it.iastate.edu/gpus) · [PyTorch](https://research.it.iastate.edu/pytorch) · [Python venvs](https://research.it.iastate.edu/python-virtual-environments)


## 0. One-time environment setup (Nova shell)

Do this **once** in a Nova Shell / terminal on a **GPU compute node** (not the login node).

```bash
# Request a GPU node
salloc -N 1 -n 4 --mem=32G --gres=gpu:1 --time=2:00:00

# Load Python tooling (exact module names can vary; use `module spider python`)
module load python-venv
# module load python/3.11
# module load py-pip

# IMPORTANT: put the venv under /work or /ptmp (NOT $HOME — quota is too small)
export PROJECT_DIR=/lustre/hdd/LAS/jernigan-lab/coms6730/HW4
mkdir -p "$PROJECT_DIR/venv" "$PROJECT_DIR/hf_cache" "$PROJECT_DIR/pip_cache"

python -m venv "$PROJECT_DIR/venv/tinyllama_ner"
source "$PROJECT_DIR/venv/tinyllama_ner/bin/activate"

python -m pip install --upgrade pip --no-cache-dir
python -m pip install --no-cache-dir torch torchvision torchaudio
python -m pip install --no-cache-dir unsloth
python -m pip install --no-cache-dir scikit-learn pandas ipykernel tqdm

# Register kernel for OnDemand Jupyter
python -m ipykernel install --user --name tinyllama_ner --display-name "TinyLlama NER (Nova GPU)"
```

Then in Jupyter: **Kernel → Change Kernel → TinyLlama NER (Nova GPU)**.

If `pip install unsloth` fails, use the fallback install notes in section 0b.


## 0b. Project paths + optional package repair

Uses `--no-cache-dir` and a work-directory cache so pip/HF do not fill `$HOME`.


In [ ]:
import os
from pathlib import Path

# Jernigan lab Lustre project path (override with env PROJECT_DIR if needed)
PROJECT_DIR = Path(
    os.environ.get("PROJECT_DIR", "/lustre/hdd/LAS/jernigan-lab/coms6730/HW4")
).expanduser()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Keep Hugging Face / pip caches off $HOME (Nova home quotas are small)
os.environ.setdefault("HF_HOME", str(PROJECT_DIR / "hf_cache"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(PROJECT_DIR / "hf_cache" / "transformers"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(PROJECT_DIR / "hf_cache" / "hub"))
os.environ.setdefault("TORCH_HOME", str(PROJECT_DIR / "hf_cache" / "torch"))
os.environ.setdefault("PIP_CACHE_DIR", str(PROJECT_DIR / "pip_cache"))
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["PIP_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR =", PROJECT_DIR)
print("HF_HOME     =", os.environ["HF_HOME"])

# Uncomment if packages are missing in this kernel:
# !pip install --no-cache-dir unsloth scikit-learn pandas tqdm
# Fallback if plain `unsloth` fails on your node:
# !pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-cache-dir --no-deps bitsandbytes accelerate peft trl triton


## 0c. Verify GPU access

If this fails, your Jupyter session likely has **no GPU**. Relaunch with **GRES = `gpu:1`**.


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU visible. Restart Jupyter OnDemand with GRES=gpu:1 "
        "and select the TinyLlama NER kernel."
    )
print("GPU:", torch.cuda.get_device_name(0))
!nvidia-smi


## 1. Locate dataset

Put `train.json`, `dev.json`, and `test.json` in a `data/` folder next to this notebook
(or set `DATA_DIR` / `HW4_DATA_DIR`).

Expected layout:

```text
HW4/
  COMS6730_HW4_TinyLlama_NER_Nova.ipynb
  data/
    train.json
    dev.json
    test.json
```

OnDemand Files uploads are limited (~20 MB). Prefer `scp` / `rsync` / Globus for the dataset.


In [ ]:
import os
import zipfile
from pathlib import Path

CANDIDATES = [
    Path("data"),
    Path.cwd() / "data",
    PROJECT_DIR / "data",
    Path.home() / "HW4" / "data",
]

DATA_DIR = None
if os.environ.get("HW4_DATA_DIR"):
    DATA_DIR = Path(os.environ["HW4_DATA_DIR"]).expanduser()
else:
    for cand in CANDIDATES:
        if (cand / "train.json").exists():
            DATA_DIR = cand
            break
    if DATA_DIR is None:
        DATA_DIR = Path("data")
        DATA_DIR.mkdir(exist_ok=True)

if Path("data.zip").exists() and not (DATA_DIR / "train.json").exists():
    with zipfile.ZipFile("data.zip", "r") as zf:
        zf.extractall(".")
    print("Extracted data.zip")

for name in ["train.json", "dev.json", "test.json"]:
    src = Path(name)
    dst = DATA_DIR / name
    if src.exists() and not dst.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        src.rename(dst)

required = ["train.json", "dev.json", "test.json"]
missing = [f for f in required if not (DATA_DIR / f).exists()]
if missing:
    raise FileNotFoundError(
        f"Missing {missing} under {DATA_DIR.resolve()}. "
        "Copy the JSON files into data/ (or set HW4_DATA_DIR / PROJECT_DIR)."
    )

print("Using DATA_DIR =", DATA_DIR.resolve())
print("Data files:", sorted(p.name for p in DATA_DIR.glob("*.json")))


## 2. Convert NER JSON → instruction-tuning (Alpaca) format

**30 pts:** Convert each example into `{instruction, input, output}` suitable for SFT.

- **Input:** sentence formed by joining `tokens`
- **Output:** disease entity spans, one per line (or `None` if none)
- Deduplicate identical spans within a sentence


In [ ]:
import json
from typing import List, Dict, Any

INSTRUCTION = (
    "Extract all disease entities from the sentence. "
    "List each unique entity span on its own line. "
    "If there are no disease entities, reply with None."
)


def tokens_to_sentence(tokens: List[str]) -> str:
    '''Join tokens with spaces (dataset already tokenizes punctuation separately).'''
    return " ".join(tokens).strip()


def entities_to_output(entities: List[Dict[str, Any]]) -> str:
    '''Format gold entities as newline-separated unique spans.'''
    spans = []
    seen = set()
    for ent in entities:
        span = (ent.get("span") or "").strip()
        if not span:
            continue
        key = span.lower()
        if key in seen:
            continue
        seen.add(key)
        spans.append(span)
    return "\n".join(spans) if spans else "None"


def load_json(path: Path) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def convert_split(examples: List[Dict[str, Any]], with_labels: bool = True) -> List[Dict[str, str]]:
    rows = []
    for ex in examples:
        sentence = tokens_to_sentence(ex.get("tokens") or [])
        # Keep empty-token examples (one exists per split) so SentIDs stay aligned
        row = {
            "id": str(ex["id"]),
            "instruction": INSTRUCTION,
            "input": sentence,
        }
        if with_labels:
            row["output"] = entities_to_output(ex.get("entities") or [])
        rows.append(row)
    return rows


train_raw = load_json(DATA_DIR / "train.json")
dev_raw = load_json(DATA_DIR / "dev.json")
test_raw = load_json(DATA_DIR / "test.json")

train_sft = convert_split(train_raw, with_labels=True)
dev_sft = convert_split(dev_raw, with_labels=True)
test_sft = convert_split(test_raw, with_labels=False)

print(f"train: {len(train_sft)} | dev: {len(dev_sft)} | test: {len(test_sft)}")
print("\n--- Example converted training sample ---")
print("instruction:", train_sft[0]["instruction"][:80], "...")
print("input:", train_sft[0]["input"])
print("output:", train_sft[0]["output"])

OUT_DIR = PROJECT_DIR / "outputs_hw4"
OUT_DIR.mkdir(parents=True, exist_ok=True)

with open(OUT_DIR / "train_sft.json", "w", encoding="utf-8") as f:
    json.dump(train_sft, f, ensure_ascii=False, indent=2)
with open(OUT_DIR / "dev_sft.json", "w", encoding="utf-8") as f:
    json.dump(dev_sft, f, ensure_ascii=False, indent=2)
print("\nSaved", OUT_DIR / "train_sft.json", "and", OUT_DIR / "dev_sft.json")


## 3. Load TinyLlama + LoRA (Unsloth)


In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 512  # NER sentences are short; 512 is enough and faster
dtype = None          # auto detect
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/tinyllama-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=False,  # set True if you OOM
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("Model + LoRA ready on", torch.cuda.get_device_name(0))


## 4. Format prompts for SFTTrainer


In [ ]:
from datasets import Dataset

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token


def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(
        examples["instruction"], examples["input"], examples["output"]
    ):
        text = alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}


# Skip empty-token rows for training (they still remain in test conversion)
train_rows = [r for r in train_sft if r["input"].strip()]
dev_rows = [r for r in dev_sft if r["input"].strip()]

train_dataset = Dataset.from_list(train_rows).map(formatting_prompts_func, batched=True)
dev_dataset = Dataset.from_list(dev_rows).map(formatting_prompts_func, batched=True)
print(f"Train rows used for SFT: {len(train_dataset)}")
print(train_dataset[0]["text"][:500])
print("...")


## 5. Fine-tune the model and save adapters

**30 pts:** Supervise-finetune TinyLlama on the converted NER dataset, then save the LoRA adapters.

On Nova A100/V100 this is often faster than Colab T4. Watch your job walltime.


In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,  # keep one NER example per sequence for clearer learning
    args=SFTConfig(
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        warmup_ratio=0.05,
        num_train_epochs=2,
        learning_rate=2e-4,
        logging_steps=20,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=str(OUT_DIR / "trainer_checkpoints"),
        report_to="none",
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataloader_num_workers=2,
    ),
)

gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

trainer_stats = trainer.train()

used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f"{trainer_stats.metrics['train_runtime']:.1f}s ({trainer_stats.metrics['train_runtime']/60:.1f} min) training.")
print(f"Peak reserved memory = {used_memory} GB.")


In [ ]:
# Save LoRA adapters under PROJECT_DIR (Lustre), not $HOME
SAVE_DIR = OUT_DIR / "tinyllama_ner_lora"
model.save_pretrained(str(SAVE_DIR))
tokenizer.save_pretrained(str(SAVE_DIR))
print(f"Saved model + tokenizer to {SAVE_DIR}")


## 6. Inference helpers + entity parsing


In [ ]:
import re
from tqdm.auto import tqdm

FastLanguageModel.for_inference(model)


def build_prompt(instruction: str, sentence: str) -> str:
    return alpaca_prompt.format(instruction, sentence, "")


def parse_entities(text: str) -> List[str]:
    '''Parse model output into unique entity spans.'''
    text = (text or "").strip()
    if "### Response:" in text:
        text = text.split("### Response:")[-1].strip()

    text = text.replace(EOS_TOKEN or "", "").strip()
    if not text or text.lower() in {"none", "n/a", "na", "null", "no entities", "none."}:
        return []

    parts = re.split(r"[\n;]+", text)
    spans = []
    seen = set()
    for part in parts:
        part = part.strip()
        part = re.sub(r"^[-\*\d\.\)\(]+\s*", "", part).strip()
        part = part.strip(" \"'`")
        if not part:
            continue
        if part.lower() in {"none", "n/a", "na", "null"}:
            continue
        key = part.lower()
        if key in seen:
            continue
        seen.add(key)
        spans.append(part)
    return spans


@torch.inference_mode()
def predict_entities(sentences: List[str], batch_size: int = 8, max_new_tokens: int = 96) -> List[List[str]]:
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"  # safer for batched causal generation

    results = []
    for i in tqdm(range(0, len(sentences), batch_size), desc="Generating"):
        batch = sentences[i : i + batch_size]
        prompts = [build_prompt(INSTRUCTION, s) for s in batch]
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_seq_length,
        ).to("cuda")
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=True,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        padded_prompt_len = inputs["input_ids"].shape[1]
        for row in outputs:
            gen = tokenizer.decode(row[padded_prompt_len:], skip_special_tokens=True)
            results.append(parse_entities(gen))
    return results


demo_inputs = [train_sft[i]["input"] for i in range(3)]
demo_preds = predict_entities(demo_inputs, batch_size=3)
for s, p, gold in zip(demo_inputs, demo_preds, train_sft[:3]):
    print("SENT:", s)
    print("PRED:", p)
    print("GOLD:", gold["output"])
    print("---")


## 7. Evaluate on development set (entity-level F1)


In [ ]:
def normalize_span(s: str) -> str:
    return re.sub(r"\s+", " ", s.strip().lower())


def entity_f1(gold_lists: List[List[str]], pred_lists: List[List[str]]) -> Dict[str, float]:
    tp = fp = fn = 0
    for gold, pred in zip(gold_lists, pred_lists):
        gset = {normalize_span(x) for x in gold if x}
        pset = {normalize_span(x) for x in pred if x}
        tp += len(gset & pset)
        fp += len(pset - gset)
        fn += len(gset - pset)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn}


dev_sentences = [r["input"] for r in dev_sft]
dev_gold = [parse_entities(r["output"]) for r in dev_sft]
dev_pred = predict_entities(dev_sentences, batch_size=8)

metrics = entity_f1(dev_gold, dev_pred)
print("Dev entity-level metrics:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

print("\nSample predictions:")
shown = 0
for s, g, p in zip(dev_sentences, dev_gold, dev_pred):
    if set(map(normalize_span, g)) != set(map(normalize_span, p)):
        print("SENT:", s)
        print("GOLD:", g)
        print("PRED:", p)
        print("---")
        shown += 1
        if shown >= 5:
            break


## 8. Predict on test set → `submission.csv`

Format:

```
ID,SentID,Span
0,4100,enzyme deficiency
1,4100,haemolysis
...
```

Each line is one entity; duplicates within a sentence are removed.


In [ ]:
import pandas as pd

test_sentences = [r["input"] for r in test_sft]
test_ids = [r["id"] for r in test_sft]
test_pred = predict_entities(test_sentences, batch_size=8)

rows = []
row_id = 0
for sent_id, spans in zip(test_ids, test_pred):
    for span in spans:
        rows.append({"ID": row_id, "SentID": sent_id, "Span": span})
        row_id += 1

submission = pd.DataFrame(rows, columns=["ID", "SentID", "Span"])
submission_path = OUT_DIR / "submission.csv"
local_submission = Path("submission.csv")
submission.to_csv(submission_path, index=False)
submission.to_csv(local_submission, index=False)
print(f"Wrote {submission_path} and {local_submission.resolve()}")
print(f"{len(submission)} entity rows covering {submission['SentID'].nunique()} sentences.")
print(submission.head(15))


## Done

Checklist:
- [x] Convert JSON → Alpaca-style SFT format
- [x] Fine-tune TinyLlama 1.1B with Unsloth LoRA and save adapters
- [x] Evaluate entity F1 on the development set
- [x] Generate Kaggle `submission.csv`

Upload `submission.csv` to: https://www.kaggle.com/competitions/coms-6730-sft-llm-challenge

Artifacts on Nova are under `PROJECT_DIR/outputs_hw4/` (LoRA adapters, converted JSON, submission).
